# FastFlow MVTec comparison

Clean reproducible baseline for comparing FastFlow with ResNet18 and DeiT backbones on MVTec AD.

Protocol:
- train only on `train/good`;
- split `train/good` into train and normal validation for loss monitoring;
- split official `test/*` into labeled validation and held-out test with defect-type stratification;
- select image-level `top_k_pixels` only on labeled validation;
- report final image metrics on held-out test with validation-selected top-k;
- save an optional test top-k sweep only as a diagnostic, not for model selection.

Run from a fresh kernel for deterministic CUDA settings to take effect before CUDA is initialized.

In [ ]:
import json
import os
import random
from copy import deepcopy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from anomalib.models.image.fastflow.loss import FastflowLoss
from anomalib.models.image.fastflow.torch_model import FastflowModel

In [ ]:
def resolve_project_root():
    root = Path.cwd().resolve()
    if (root / "datasets").exists() and (root / "experiments").exists():
        return root
    if (root.parent / "datasets").exists() and (root.parent / "experiments").exists():
        return root.parent
    raise FileNotFoundError("Cannot resolve project root with datasets/ and experiments/ directories")


PROJECT_ROOT = resolve_project_root()
DATASET_ROOT = PROJECT_ROOT / "datasets" / "MVTecAD"
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"

CLASS_NAMES = ["metal_nut", "hazelnut"]
DEFECT_FILTERS = {
    "metal_nut": ["good", "bent", "scratch", "color"],
    "hazelnut": None,
}

TRY_NUMBER = 1
RANDOM_SEEDS = [42]
BACKBONES_TO_RUN = ["resnet18_384", "deit_base_distilled_384"]

RUN_TRAINING = True
RUN_EVAL = True
ALLOW_OVERWRITE = False
SAVE_TEST_TOP_K_SWEEP_DIAGNOSTIC = True

TRAIN_GOOD_VAL_FRACTION = 0.1
LABELED_VAL_FRACTION = 0.5
THRESHOLD_PERCENTILE = 95
NUM_WORKERS = 0
DETERMINISTIC_TRAINING = True
USE_TRAIN_AUGMENTATION = False

BACKBONE_CONFIGS = {
    # Same 384 input size as DeiT to isolate backbone effects from resolution effects.
    "resnet18_384": {
        "tag": "fastflow_resnet18_384",
        "backbone": "resnet18",
        "image_size": 384,
        "flow_steps": 10,
        "hidden_ratio": None,
        "num_epochs": 30,
        "learning_rate": 1e-3,
        "weight_decay": 1e-3,
        "eta_min": 1e-6,
        "batch_size": 8,
        "grad_clip_norm": None,
    },
    "deit_base_distilled_384": {
        "tag": "fastflow_deit_base_distilled_384",
        "backbone": "deit_base_distilled_patch16_384",
        "image_size": 384,
        "flow_steps": 8,
        "hidden_ratio": 0.5,
        "num_epochs": 40,
        "learning_rate": 3e-5,
        "weight_decay": 1e-5,
        "eta_min": 1e-6,
        "batch_size": 8,
        "grad_clip_norm": 10.0,
    },
}

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"DATASET_ROOT={DATASET_ROOT}")

In [ ]:
def seed_everything(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_generator(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


seed_everything(RANDOM_SEEDS[0], deterministic=DETERMINISTIC_TRAINING)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])


def get_image_transform(image_size, train=False, augment=False):
    steps = [transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BICUBIC)]
    if train and augment:
        steps.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
        ])
    steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return transforms.Compose(steps)


def get_mask_transform(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), interpolation=InterpolationMode.NEAREST),
        transforms.ToTensor(),
    ])


def find_mask_path(class_dir, defect_type, image_path):
    gt_dir = Path(class_dir) / "ground_truth" / defect_type
    preferred = gt_dir / f"{Path(image_path).stem}_mask.png"
    if preferred.exists():
        return preferred
    matches = sorted(gt_dir.glob(f"{Path(image_path).stem}_mask.*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Mask not found for {image_path}")


def collect_mvtec_samples(dataset_root, class_name, defect_filter=None):
    class_dir = Path(dataset_root) / class_name
    test_dir = class_dir / "test"
    if not test_dir.exists():
        raise FileNotFoundError(test_dir)

    allowed = None if defect_filter is None else set(defect_filter)
    samples = []
    for defect_dir in sorted([p for p in test_dir.iterdir() if p.is_dir()]):
        defect_type = defect_dir.name
        if allowed is not None and defect_type not in allowed:
            continue
        label = int(defect_type != "good")
        for image_path in list_images(defect_dir):
            mask_path = None
            if label == 1:
                mask_path = find_mask_path(class_dir, defect_type, image_path)
            samples.append({
                "image_path": str(image_path),
                "mask_path": None if mask_path is None else str(mask_path),
                "label": label,
                "defect_type": defect_type,
            })
    return samples


def split_labeled_samples(samples, seed, val_fraction):
    strata = [sample["defect_type"] for sample in samples]
    try:
        val_samples, test_samples = train_test_split(
            samples,
            test_size=1.0 - val_fraction,
            random_state=seed,
            shuffle=True,
            stratify=strata,
        )
    except ValueError:
        labels = [sample["label"] for sample in samples]
        val_samples, test_samples = train_test_split(
            samples,
            test_size=1.0 - val_fraction,
            random_state=seed,
            shuffle=True,
            stratify=labels,
        )
    return list(val_samples), list(test_samples)


def split_train_good_paths(dataset_root, class_name, seed, val_fraction):
    paths = list_images(Path(dataset_root) / class_name / "train" / "good")
    train_paths, val_paths = train_test_split(
        paths,
        test_size=val_fraction,
        random_state=seed,
        shuffle=True,
    )
    return list(train_paths), list(val_paths)


class ImagePathDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = [str(path) for path in image_paths]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(image)


class MVTecEvalDataset(Dataset):
    def __init__(self, samples, image_transform, mask_transform=None):
        self.samples = list(samples)
        self.image_transform = image_transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        image = self.image_transform(image)
        mask = None
        if sample["mask_path"] is not None:
            mask = Image.open(sample["mask_path"]).convert("L")
            mask = self.mask_transform(mask) if self.mask_transform else transforms.ToTensor()(mask)
        return image, sample["label"], sample["defect_type"], sample["image_path"], mask

In [ ]:
def freeze_feature_extractor(model):
    frozen_params_num = 0
    if hasattr(model, "feature_extractor"):
        for param in model.feature_extractor.parameters():
            param.requires_grad = False
            frozen_params_num += param.numel()
    print(f"Parameters frozen: {frozen_params_num}")


def get_trainable_parameters(model):
    trainable_params = []
    trainable_params_num = 0
    for param in model.parameters():
        if param.requires_grad:
            trainable_params.append(param)
            trainable_params_num += param.numel()
    print(f"Trainable parameters: {trainable_params_num}")
    return trainable_params


def build_fastflow_model(config):
    kwargs = {
        "backbone": config["backbone"],
        "flow_steps": config["flow_steps"],
        "input_size": [config["image_size"], config["image_size"]],
        "pre_trained": True,
    }
    if config.get("hidden_ratio") is not None:
        kwargs["hidden_ratio"] = config["hidden_ratio"]
    return FastflowModel(**kwargs)


def make_loader(dataset, batch_size, shuffle, seed, num_workers=0):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=seed_worker,
        generator=make_generator(seed),
    )


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_text_metrics(path, metrics):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for key, value in metrics.items():
            f.write(f"{key}: {value}\n")


In [ ]:

def compute_fastflow_loss(criterion, outputs):
    hidden_variables = getattr(outputs, "hidden_variables", None)
    jacobians = getattr(outputs, "jacobians", None)

    if isinstance(outputs, dict):
        hidden_variables = outputs.get("hidden_variables", hidden_variables)
        jacobians = outputs.get("jacobians", jacobians)

    if hidden_variables is None or jacobians is None:
        if isinstance(outputs, (tuple, list)) and len(outputs) >= 2:
            hidden_variables, jacobians = outputs[0], outputs[1]

    if hidden_variables is None or jacobians is None:
        output_type = type(outputs).__name__
        available = [name for name in dir(outputs) if not name.startswith("_")]
        raise TypeError(
            "FastflowModel did not return hidden_variables/jacobians needed for FastflowLoss. "
            f"output_type={output_type}, available_attrs={available[:20]}"
        )

    return criterion(hidden_variables, jacobians)

def train_fastflow(
    model,
    train_dataset,
    normal_val_dataset,
    config,
    log_dir,
    seed,
    device,
    num_workers=0,
    freeze_extractor=True,
):
    model = model.to(device)
    if freeze_extractor:
        freeze_feature_extractor(model)
    trainable_params = get_trainable_parameters(model)

    criterion = FastflowLoss()
    optimizer = optim.AdamW(
        trainable_params,
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    train_loader = make_loader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        seed=seed,
        num_workers=num_workers,
    )
    val_loader = make_loader(
        normal_val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        seed=seed,
        num_workers=num_workers,
    )

    total_steps = max(1, config["num_epochs"] * len(train_loader))
    warmup_steps = max(1, int(0.1 * total_steps))
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.05,
        end_factor=1.0,
        total_iters=warmup_steps,
    )
    main_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, total_steps - warmup_steps),
        eta_min=config["eta_min"],
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, main_scheduler],
        milestones=[warmup_steps],
    )

    history = {"train_loss": [], "normal_val_loss": []}
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(config["num_epochs"]):
        model.train()
        train_losses = []
        for images in tqdm(train_loader, desc=f"train {epoch + 1}/{config['num_epochs']}", leave=False):
            images = images.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = compute_fastflow_loss(criterion, outputs)
            loss.backward()
            if config.get("grad_clip_norm") is not None:
                torch.nn.utils.clip_grad_norm_(trainable_params, config["grad_clip_norm"])
            optimizer.step()
            scheduler.step()
            train_losses.append(float(loss.detach().cpu()))

        # FastFlow loss needs train-mode outputs: hidden_variables and jacobians.
        model.train()
        val_losses = []
        with torch.no_grad():
            for images in tqdm(val_loader, desc=f"normal-val {epoch + 1}/{config['num_epochs']}", leave=False):
                images = images.to(device)
                outputs = model(images)
                loss = compute_fastflow_loss(criterion, outputs)
                val_losses.append(float(loss.detach().cpu()))

        train_loss = float(np.mean(train_losses))
        normal_val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["normal_val_loss"].append(normal_val_loss)
        print(f"epoch={epoch + 1:03d} train_loss={train_loss:.6f} normal_val_loss={normal_val_loss:.6f}")

        if normal_val_loss < best_val_loss:
            best_val_loss = normal_val_loss
            best_state = deepcopy(model.state_dict())

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "config": config,
            "seed": seed,
        }
        torch.save(checkpoint, Path(log_dir) / "last_checkpoint.pth")

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save(model.state_dict(), Path(log_dir) / "trained_model_weights.pth")
    torch.save(model, Path(log_dir) / "trained_model.pth")
    pd.DataFrame(history).to_csv(Path(log_dir) / "train_history.csv", index=False)
    save_json(Path(log_dir) / "training_summary.json", {
        "best_normal_val_loss": best_val_loss,
        "final_train_loss": history["train_loss"][-1],
        "final_normal_val_loss": history["normal_val_loss"][-1],
    })
    return model, history

In [ ]:
def anomaly_map_for_image(model, image, device):
    model.eval()
    with torch.no_grad():
        outputs = model(image.unsqueeze(0).to(device))
        anomaly_map = outputs.anomaly_map.detach().cpu().float().squeeze()
    if anomaly_map.ndim == 3:
        anomaly_map = anomaly_map[0]
    if anomaly_map.ndim != 2:
        raise ValueError(f"Expected 2D anomaly map, got shape={tuple(anomaly_map.shape)}")
    return anomaly_map


def resize_map(anomaly_map, size_hw):
    if tuple(anomaly_map.shape[-2:]) == tuple(size_hw):
        return anomaly_map
    x = anomaly_map.unsqueeze(0).unsqueeze(0)
    x = F.interpolate(x, size=size_hw, mode="bilinear", align_corners=False)
    return x.squeeze(0).squeeze(0)


def make_top_k_grid(num_pixels):
    fractions = [0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.20, 1.0 / 3.0, 0.50, 1.0]
    absolute = [1, 4, 16, 64, 147, 256, 1024, 2048, 4096, 8192, 16384, 32768, 49152]
    candidates = set(absolute)
    candidates.update(int(round(num_pixels * fraction)) for fraction in fractions)
    return sorted(k for k in candidates if 1 <= k <= num_pixels)


def score_anomaly_map(anomaly_map, top_k_pixels):
    values = anomaly_map.flatten()
    k = max(1, min(int(top_k_pixels), values.numel()))
    return float(torch.topk(values, k).values.mean())


def compute_cohens_d(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=float)
    normal = scores[labels == 0]
    anomaly = scores[labels == 1]
    if len(normal) < 2 or len(anomaly) < 2:
        return float("nan")
    pooled = np.sqrt(((len(normal) - 1) * normal.var(ddof=1) + (len(anomaly) - 1) * anomaly.var(ddof=1)) / (len(normal) + len(anomaly) - 2))
    if pooled == 0:
        return float("nan")
    return float((anomaly.mean() - normal.mean()) / pooled)


def compute_image_metrics(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    metrics = {
        "image_roc_auc": float(roc_auc_score(labels, scores)),
        "image_average_precision": float(average_precision_score(labels, scores)),
        "cohens_d": compute_cohens_d(labels, scores),
        "normal_score_mean": float(scores[labels == 0].mean()),
        "anomaly_score_mean": float(scores[labels == 1].mean()),
        "num_normal": int((labels == 0).sum()),
        "num_anomaly": int((labels == 1).sum()),
    }
    return metrics


def compute_threshold_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": float(fp / max(1, fp + tn)),
        "fnr": float(fn / max(1, fn + tp)),
    }


def sweep_top_k(model, dataset, top_k_grid, device, desc):
    labels = []
    defects = []
    paths = []
    scores_by_k = {int(k): [] for k in top_k_grid}

    for idx in tqdm(range(len(dataset)), desc=desc, leave=False):
        image, label, defect_type, image_path, _ = dataset[idx]
        anomaly_map = anomaly_map_for_image(model, image, device)
        labels.append(int(label))
        defects.append(defect_type)
        paths.append(image_path)
        for top_k in top_k_grid:
            scores_by_k[int(top_k)].append(score_anomaly_map(anomaly_map, top_k))

    metric_rows = []
    num_pixels = int(dataset[0][0].shape[-1] * dataset[0][0].shape[-2])
    for top_k, scores in scores_by_k.items():
        row = {
            "top_k_pixels": int(top_k),
            "top_k_fraction": float(top_k / num_pixels),
        }
        row.update(compute_image_metrics(labels, scores))
        metric_rows.append(row)

    metrics_df = pd.DataFrame(metric_rows).sort_values("top_k_pixels").reset_index(drop=True)
    cache = {
        "labels": labels,
        "defects": defects,
        "paths": paths,
        "scores_by_k": scores_by_k,
    }
    return metrics_df, cache


def select_top_k(val_sweep_df):
    ranked = val_sweep_df.sort_values(
        ["image_roc_auc", "image_average_precision", "cohens_d", "top_k_pixels"],
        ascending=[False, False, False, True],
    )
    return int(ranked.iloc[0]["top_k_pixels"]), ranked.iloc[0].to_dict()


def make_score_rows(cache, selected_top_k):
    relative_paths = [Path(path).resolve().relative_to(PROJECT_ROOT).as_posix() for path in cache["paths"]]
    return pd.DataFrame({
        "image_path": relative_paths,
        "defect_type": cache["defects"],
        "label": cache["labels"],
        "score": cache["scores_by_k"][int(selected_top_k)],
    })


def evaluate_pixel_metrics(model, dataset, device, desc):
    y_true_chunks = []
    y_score_chunks = []
    num_images = 0
    for idx in tqdm(range(len(dataset)), desc=desc, leave=False):
        image, label, _, _, mask = dataset[idx]
        if int(label) == 0 or mask is None:
            continue
        anomaly_map = anomaly_map_for_image(model, image, device)
        mask_2d = mask.squeeze().float()
        anomaly_map = resize_map(anomaly_map, mask_2d.shape[-2:])
        y_true_chunks.append((mask_2d.flatten().numpy() > 0.5).astype(np.uint8))
        y_score_chunks.append(anomaly_map.flatten().numpy().astype(np.float32))
        num_images += 1

    if not y_true_chunks:
        return {"pixel_roc_auc": float("nan"), "pixel_average_precision": float("nan"), "pixel_num_images": 0}

    y_true = np.concatenate(y_true_chunks)
    y_score = np.concatenate(y_score_chunks)
    return {
        "pixel_roc_auc": float(roc_auc_score(y_true, y_score)),
        "pixel_average_precision": float(average_precision_score(y_true, y_score)),
        "pixel_num_images": int(num_images),
    }

In [ ]:
def has_experiment_artifacts(log_dir):
    artifact_names = {
        "last_checkpoint.pth",
        "trained_model.pth",
        "trained_model_weights.pth",
        "train_history.csv",
        "training_summary.json",
        "val_top_k_sweep_metrics.csv",
        "test_top_k_sweep_metrics_DIAGNOSTIC_ONLY.csv",
        "val_scores.csv",
        "test_scores.csv",
        "test_metrics.json",
        "test_metrics.txt",
    }
    return any((Path(log_dir) / name).exists() for name in artifact_names)


def prepare_experiment_dir(class_name, config, try_number, seed):
    log_dir = EXPERIMENTS_ROOT / class_name / config["tag"] / f"try_{try_number}_seed_{seed}"
    if RUN_TRAINING and log_dir.exists() and any(log_dir.iterdir()) and not ALLOW_OVERWRITE:
        if has_experiment_artifacts(log_dir):
            raise FileExistsError(f"Refusing to overwrite experiment artifacts: {log_dir}")
        print(f"Reusing config-only/incomplete experiment directory: {log_dir}")
    log_dir.mkdir(parents=True, exist_ok=True)
    return log_dir


def load_trained_model(config, model_path, device):
    model = build_fastflow_model(config)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def run_one_experiment(class_name, config_name, seed):
    seed_everything(seed, deterministic=DETERMINISTIC_TRAINING)
    config = BACKBONE_CONFIGS[config_name]
    log_dir = prepare_experiment_dir(class_name, config, TRY_NUMBER, seed)
    relative_log_dir = log_dir.resolve().relative_to(PROJECT_ROOT).as_posix()
    image_size = config["image_size"]

    train_transform = get_image_transform(image_size, train=True, augment=USE_TRAIN_AUGMENTATION)
    eval_transform = get_image_transform(image_size, train=False, augment=False)
    mask_transform = get_mask_transform(image_size)

    train_paths, normal_val_paths = split_train_good_paths(DATASET_ROOT, class_name, seed, TRAIN_GOOD_VAL_FRACTION)
    labeled_samples = collect_mvtec_samples(DATASET_ROOT, class_name, DEFECT_FILTERS.get(class_name))
    labeled_val_samples, labeled_test_samples = split_labeled_samples(labeled_samples, seed, LABELED_VAL_FRACTION)

    train_dataset = ImagePathDataset(train_paths, train_transform)
    normal_val_dataset = ImagePathDataset(normal_val_paths, eval_transform)
    labeled_val_dataset = MVTecEvalDataset(labeled_val_samples, eval_transform, mask_transform)
    labeled_test_dataset = MVTecEvalDataset(labeled_test_samples, eval_transform, mask_transform)

    split_summary = {
        "train_good": len(train_dataset),
        "normal_val_good": len(normal_val_dataset),
        "labeled_val": len(labeled_val_dataset),
        "labeled_test": len(labeled_test_dataset),
        "labeled_val_by_type": pd.Series([s["defect_type"] for s in labeled_val_samples]).value_counts().sort_index().to_dict(),
        "labeled_test_by_type": pd.Series([s["defect_type"] for s in labeled_test_samples]).value_counts().sort_index().to_dict(),
    }

    run_config = {
        "class_name": class_name,
        "config_name": config_name,
        "seed": seed,
        "try_number": TRY_NUMBER,
        "config": config,
        "split_summary": split_summary,
        "top_k_selection": "validation image_roc_auc, tie by AP/cohens_d/smaller k",
        "threshold_percentile": THRESHOLD_PERCENTILE,
    }
    save_json(log_dir / "run_config.json", run_config)
    print(json.dumps({"log_dir": relative_log_dir, **split_summary}, ensure_ascii=False, indent=2))

    model_path = log_dir / "trained_model_weights.pth"
    if RUN_TRAINING:
        model = build_fastflow_model(config)
        model, history = train_fastflow(
            model,
            train_dataset,
            normal_val_dataset,
            config,
            log_dir,
            seed,
            device,
            num_workers=NUM_WORKERS,
        )
    else:
        if not model_path.exists():
            raise FileNotFoundError(model_path)
        model = load_trained_model(config, model_path, device)

    if not RUN_EVAL:
        return {"class_name": class_name, "config_name": config_name, "seed": seed, "log_dir": relative_log_dir}

    num_pixels = image_size * image_size
    top_k_grid = make_top_k_grid(num_pixels)

    val_sweep_df, val_cache = sweep_top_k(model, labeled_val_dataset, top_k_grid, device, desc=f"val top-k {class_name} {config_name}")
    selected_top_k, selected_val_row = select_top_k(val_sweep_df)
    val_sweep_df["selected"] = val_sweep_df["top_k_pixels"] == selected_top_k
    val_sweep_df.to_csv(log_dir / "val_top_k_sweep_metrics.csv", index=False)

    val_rows = make_score_rows(val_cache, selected_top_k)
    val_rows.to_csv(log_dir / "val_scores.csv", index=False)
    val_scores = val_rows["score"].to_numpy(dtype=float)
    val_labels = val_rows["label"].to_numpy(dtype=int)
    threshold = float(np.percentile(val_scores[val_labels == 0], THRESHOLD_PERCENTILE))
    val_image_metrics = compute_image_metrics(val_labels, val_scores)
    val_threshold_metrics = compute_threshold_metrics(val_labels, val_scores, threshold)

    test_sweep_df, test_cache = sweep_top_k(model, labeled_test_dataset, top_k_grid, device, desc=f"test top-k {class_name} {config_name}")
    if SAVE_TEST_TOP_K_SWEEP_DIAGNOSTIC:
        test_sweep_df.to_csv(log_dir / "test_top_k_sweep_metrics_DIAGNOSTIC_ONLY.csv", index=False)

    test_rows = make_score_rows(test_cache, selected_top_k)
    test_rows.to_csv(log_dir / "test_scores.csv", index=False)
    test_scores = test_rows["score"].to_numpy(dtype=float)
    test_labels = test_rows["label"].to_numpy(dtype=int)
    test_image_metrics = compute_image_metrics(test_labels, test_scores)
    test_threshold_metrics = compute_threshold_metrics(test_labels, test_scores, threshold)
    test_pixel_metrics = evaluate_pixel_metrics(model, labeled_test_dataset, device, desc=f"test pixel {class_name} {config_name}")

    final_metrics = {
        "class_name": class_name,
        "config_name": config_name,
        "seed": seed,
        "selected_top_k_pixels": int(selected_top_k),
        "selected_top_k_fraction": float(selected_top_k / num_pixels),
        "val_selected_image_roc_auc": float(selected_val_row["image_roc_auc"]),
        "val_selected_image_average_precision": float(selected_val_row["image_average_precision"]),
        "val_image_roc_auc": val_image_metrics["image_roc_auc"],
        "val_image_average_precision": val_image_metrics["image_average_precision"],
        "val_balanced_accuracy_at_threshold": val_threshold_metrics["balanced_accuracy"],
        "test_image_roc_auc": test_image_metrics["image_roc_auc"],
        "test_image_average_precision": test_image_metrics["image_average_precision"],
        "test_cohens_d": test_image_metrics["cohens_d"],
        "test_threshold": test_threshold_metrics["threshold"],
        "test_balanced_accuracy_at_val_threshold": test_threshold_metrics["balanced_accuracy"],
        "test_fpr_at_val_threshold": test_threshold_metrics["fpr"],
        "test_fnr_at_val_threshold": test_threshold_metrics["fnr"],
        **test_pixel_metrics,
    }
    save_json(log_dir / "test_metrics.json", final_metrics)
    save_text_metrics(log_dir / "test_metrics.txt", final_metrics)
    print(json.dumps(final_metrics, ensure_ascii=False, indent=2))
    return {**final_metrics, "log_dir": relative_log_dir}

In [ ]:
summary_rows = []
for seed in RANDOM_SEEDS:
    for class_name in CLASS_NAMES:
        for config_name in BACKBONES_TO_RUN:
            summary_rows.append(run_one_experiment(class_name, config_name, seed))

summary_df = pd.DataFrame(summary_rows)
summary_path = EXPERIMENTS_ROOT / f"mvtec_fastflow_summary_try_{TRY_NUMBER}.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
display(summary_df)

## How to interpret top-k files

Use `val_top_k_sweep_metrics.csv` for model selection. The final `test_metrics.txt/json` uses only the validation-selected `selected_top_k_pixels`.

`test_top_k_sweep_metrics_DIAGNOSTIC_ONLY.csv` is intentionally named as diagnostic: it is useful for error analysis and checking transfer of top-k, but it must not be used to choose the final top-k for reported test quality.